In [1]:
import torch
from torch import nn
from torch.nn import functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")
with open('/kaggle/input/datasets/lifatsas/my-train/zh_wiki_clean.txt', "r", encoding = 'utf-8') as f:
    text = f.read()
tokens = list(text)

vocab_size = tok.vocab_size
block_size = 32
batch_size = 64
n_embd = 128
n_head = 4
layer_n = 3
train_loop = 10000
head_size = n_embd // n_head

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [2]:
def encode(text):
    encoded = tok.encode(text)
    return encoded

ids = encode(text[50000:])

def decode(ids):
    decoded = tok.decode(ids)
    return decoded
    

print(f"raw data length {len(tokens)}")
print(f"new data length {len(ids)}")
print(f"ration = {len(tokens)/len(ids):.2f}x")

Token indices sequence length is longer than the specified maximum sequence length for this model (59728012 > 131072). Running this sequence through the model will result in indexing errors


raw data length 86167254
new data length 59728012
ration = 1.44x


In [3]:
data = torch.tensor(ids, dtype = torch.long)
n = int(len(data) * 0.9)
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint((len(data) - block_size - 1), (batch_size, ))
    x = torch.stack([data[i:i + block_size] for i in ix]).to(device)
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix]).to(device)
    return x, y

In [4]:
@torch.no_grad()
def estimate_loss(step):
    model.eval()
    results = {}
    for split in ['train', 'val']:
        losses = []
        for _ in range(50):
            x, y = get_batch(split)
            _, loss = model(x, y)
            losses.append(loss.mean().item())
        results[split] = sum(losses) / len(losses)
    print(f"step {step} | train loss: {results['train']:.4f} | val loss: {results['val']:.4f}")
    model.train()

def rope(x):
    B, T, head_size = x.shape
    position = torch.arange(T, device=device).unsqueeze(1) # T, 1
    dim = torch.arange(0, head_size, 2, device=device) # head_size/2
    theta = (1 / 10000**(dim/head_size)) # head_size/2
    angles = (theta * position) # T, head_size/2

    x1 = x[..., 0::2]
    x2 = x[..., 1::2]

    x_rope = torch.stack([
        x1 * torch.cos(angles) - x2 * torch.sin(angles) , # B, T, head_size/2, 2
        x1 * torch.sin(angles) + x2 * torch.cos(angles),
    ], dim=-1).flatten(-2) # B, T, head_size/2 * 2

    return x_rope #B, T, head_size

class Head(nn.Module):
    def __init__(self, n_embd, head_size):
        super().__init__()
        self.W_q = nn.Linear(n_embd, head_size, bias = False) #take n_embd(how many)
        self.W_k = nn.Linear(n_embd, head_size, bias = False) #to n_head(how distinct)
        self.W_v = nn.Linear(n_embd, head_size, bias = False) #each one has head_size(how large)

    def forward(self, x): # x = B, T, n_embd
        T = x.shape[-2]

        w_q = self.W_q(x) # B, T, head_size
        w_k = self.W_k(x)
        w_v = self.W_v(x)

        w_q = rope(w_q)
        w_k = rope(w_k)

        weight = w_q @ w_k.transpose(-2, -1) # B, T, head_size @ B, head_size, T -> B, T, T
        weight = weight * (head_size**(-0.5)) # prevent softmax collapses 
        tril = torch.tril(torch.ones(T, T, device=x.device))
        weight = weight.masked_fill(tril == 0, float('-inf'))
        weight = torch.softmax(weight, dim=-1)
        out = weight @ w_v # -> B, T, head_size

        return out # B, T, head_size
    
class MultiHead(nn.Module):
    def __init__(self, n_head, head_size, n_embd):
        super().__init__()
        self.att = nn.ModuleList([Head(n_embd, head_size) for _ in range(n_head)])
        self.proj = nn.Linear(n_head * head_size, n_embd)
    
    def forward(self, x): # x = B, T, n_embd
        out = torch.cat([h(x) for h in self.att], dim=-1) # B, T,  n_head * head_size
        return self.proj(out) # B, T, n_embd
    
class FeedFoward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4*n_embd),
            nn.ReLU(),
            nn.Linear(4*n_embd, n_embd),
        )
    
    def forward(self, x):
        return self.net(x) 
    
class Block(nn.Module):
    def __init__(self, n_head, head_size, n_embd):
        super().__init__()
        self.att = MultiHead(n_head, head_size, n_embd)
        self.ffn = FeedFoward(n_embd)
        self.drop = nn.Dropout(p=0.2)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self,x): # x = B, T, n_embd 
        x = x + self.drop(self.att(self.ln1(x))) # residual + attention 
        x = x + self.drop(self.ffn(self.ln2(x))) # residual + ffn
        return x

In [5]:
class My_transmodel(nn.Module):
    def __init__(self, n_head, n_embd, head_size,layer_n):
        super().__init__()
        self.embedding_table = nn.Embedding(vocab_size, n_embd)
        self.block = nn.Sequential(*[Block(n_head, head_size, n_embd) for _ in range(layer_n)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.project_out = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets = None):
        logits = self.embedding_table(idx)
        logits = self.block(logits)
        logits = self.ln_f(logits)
        logits = self.project_out(logits)

        if targets is not None:
            B, T, C =logits.shape
            logits = logits.view(B*T, C) # loss function need N(all the tokens be flattened), C not B, T, C
            targets = targets.view(B*T,)
            loss = F.cross_entropy(logits, targets)
        else:
            loss = None

        return logits, loss

    
    def generate(self, ids, max_netx_tokens):
        for _ in range(max_netx_tokens):
            idx_cond = ids[: , -block_size:] # B, T
            logits, _ = self(idx_cond)
            logits = logits[: ,-1, :]
            prob = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(prob, num_samples=1)
            ids = torch.cat([ids, idx_next], dim=1)
        return ids

In [6]:
model = My_transmodel(n_head, n_embd, head_size, layer_n).to(device)
model = nn.DataParallel(model) # multi gpu train
optimzer = torch.optim.Adam(model.parameters(), lr=1e-3)
total_paremeters = sum(p.numel() for p in model.parameters())
print(f'paremeters:{total_paremeters/1e6}M')

for step in range(train_loop):
    if step % 200 == 0:
        estimate_loss(step)
   
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    loss = loss.mean() # the dataprallet given verctor per gpu not scalar
    optimzer.zero_grad(set_to_none=True)
    loss.backward()
    optimzer.step()
    
print(f"final evaluation: {estimate(train_loop)}")

paremeters:39.566171M


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


step 0 | train loss: 12.1105 | val loss: 12.1128
step 200 | train loss: 7.5108 | val loss: 7.5622
step 400 | train loss: 7.1299 | val loss: 7.1850
step 600 | train loss: 6.9079 | val loss: 7.0266
step 800 | train loss: 6.7709 | val loss: 6.8595
step 1000 | train loss: 6.6436 | val loss: 6.7913
step 1200 | train loss: 6.5324 | val loss: 6.6846
step 1400 | train loss: 6.4370 | val loss: 6.5737
step 1600 | train loss: 6.3888 | val loss: 6.5122
step 1800 | train loss: 6.2940 | val loss: 6.4271
step 2000 | train loss: 6.1884 | val loss: 6.4540
step 2200 | train loss: 6.1715 | val loss: 6.3365
step 2400 | train loss: 6.1837 | val loss: 6.3364
step 2600 | train loss: 6.1135 | val loss: 6.2617
step 2800 | train loss: 6.0386 | val loss: 6.2323
step 3000 | train loss: 6.0186 | val loss: 6.2358
step 3200 | train loss: 6.0080 | val loss: 6.1906
step 3400 | train loss: 5.9792 | val loss: 6.1419
step 3600 | train loss: 5.9138 | val loss: 6.1362
step 3800 | train loss: 5.9147 | val loss: 6.0636
step 

NameError: name 'estimate' is not defined

In [ ]:
# for step in range(train_loop):
#     if step % 200 == 0:
#         estimate_loss(step)
   
#     xb, yb = get_batch('train')
#     logits, loss = model(xb, yb)
#     loss = loss.mean() # the dataprallet given verctor per gpu not scalar
#     optimzer.zero_grad(set_to_none=True)
#     loss.backward()
#     optimzer.step()
    
# print(f"final evaluation: {estimate(train_loop)}")

In [ ]:
text = '你'
ids = encode(text)
ids = torch.tensor(ids, dtype=torch.long).unsqueeze(0).to(device)
ids = model.module.generate(ids, max_netx_tokens=100)
print(decode(ids[0].tolist()))

In [ ]:
torch.save(model.state_dict(), 'my_transformer_zh_wiki.pt')